# Notes

Perform logistic regression at residue level but only focus on entangled proteins.

This notebook analysis control the SASA as the confounder. 
Reason:

change in entanglement -> change in SASA
change in SASA -> change in proteolytic susceptibility.

Therefore, SASA is a mediator, not confounder. To estimate the total effect of entangled region on proteolytic susceptibility, we should not include the SASA as the confounder.

In [1]:
import pandas as pd
# import pandas as pd
import numpy as np
import scipy.stats as ss
from statsmodels.stats.contingency_tables import Table2x2
import statsmodels.formula.api as smf

In [2]:
def fmt_p(p):
    """Mixed formatting for p-values."""
    return f"{p:.3e}" if p < 0.001 else f"{p:.3f}"

def summarize_logit(result, alpha=0.05, exponentiate=True):
    """
    Summarize statsmodels Logit/GLM(Binomial) results.

    Returns a DataFrame with:
    coef, OR, CI, numeric p-value, and formatted p-value.
    """
    params = result.params
    conf = result.conf_int(alpha=alpha)   # columns: [lower, upper]
    pvals = result.pvalues

    df = pd.DataFrame({
        "coef": params,
        "ci_lower": conf.iloc[:, 0],
        "ci_upper": conf.iloc[:, 1],
        "pvalue": pvals
    })

    if exponentiate:
        df["OR"] = np.exp(df["coef"])
        df["OR_ci_lower"] = np.exp(df["ci_lower"])
        df["OR_ci_upper"] = np.exp(df["ci_upper"])

    # add formatted p-value column
    df["p_fmt"] = df["pvalue"].apply(fmt_p)

    # nicer column order
    if exponentiate:
        df = df[[
            "coef",
            "OR",
            "OR_ci_lower",
            "OR_ci_upper",
            "pvalue",
            "p_fmt"
        ]]
    else:
        df = df[["coef", "ci_lower", "ci_upper", "pvalue", "p_fmt"]]

    return df

In [3]:
df_all = pd.read_pickle('../data/SC_Ent.pkl')
# Only select entangled proteins
df = df_all[df_all['entangled']==1]
df

,SGDID,Uniprot,SC,length_AF,entangled,asphericity,fraction_idp,Knot,cov_lasso,entangled_residues,Cutsite_residues,clustered_entangled_residues
0,S000002468,Q12298,0,539,1,0.139968,0.011132,0,0,"[26, 27, 28, 31, 33, 34, 35, 36, 37, 38, 39, 4...",[],"[27, 31, 34, 35, 36, 38, 39, 40, 41, 42, 43, 4..."
1,S000000094,P35842,0,467,1,0.097823,0.051392,0,0,"[21, 23, 30, 31, 32, 33, 34, 35, 37, 38, 41, 4...",[],"[32, 33, 34, 35, 37, 38, 41, 46, 47, 67, 68, 6..."
2,S000002490,P38961,0,392,1,0.108081,0.295918,0,0,"[177, 178, 179, 180, 181, 182, 183, 184, 190, ...",[],"[181, 193, 266, 267, 268, 290, 292, 293, 294, ..."
3,S000003653,P46956,0,311,1,0.311604,0.099678,0,0,"[64, 65, 68, 69, 70, 71, 72, 73, 74, 75, 78, 1...",[],"[203, 210, 211, 212, 213, 252, 253, 254, 255, ..."
5,S000003242,P53204,0,395,1,0.102152,0.293671,0,0,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 15, 16, 17...",[],"[1, 2, 3, 4, 5, 6, 7, 8, 9, 15, 16, 17, 18, 19..."
...,...,...,...,...,...,...,...,...,...,...,...,...
2249,S000004850,Q05029,0,724,1,0.111921,0.084254,0,0,"[5, 6, 7, 8, 9, 10, 11, 12, 15, 16, 18, 19, 23...",[],"[6, 7, 8, 9, 10, 11, 12, 15, 35, 37, 38, 39, 4..."
2250,S000005337,P53743,0,316,1,0.212209,0.506329,0,0,"[103, 104, 105, 106, 107, 108, 109, 110, 111, ...",[],"[103, 104, 105, 106, 107, 108, 109, 110, 111, ..."
2251,S000001943,P43619,0,295,1,0.383145,0.071186,0,0,"[1, 2, 3, 4, 5, 8, 10, 12, 13, 14, 15, 16, 17,...",[],"[15, 18, 19, 20, 21, 22, 23, 24, 25, 26, 48, 4..."
2254,S000005185,P11412,1,505,1,0.142002,0.017822,0,0,"[6, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, ...","[394, 395, 396, 397, 398, 399, 400, 401, 402]","[10, 11, 12, 13, 14, 15, 16, 17, 18, 23, 24, 2..."


In [4]:
# Standardize 'length_AF' using z-score
df['length_z'] = (df['length_AF'] - df['length_AF'].mean()) / df['length_AF'].std()
df

/var/folders/fp/t9px_8vd6639966sxyppg_g00000gn/T/ipykernel_74338/2228520070.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['length_z'] = (df['length_AF'] - df['length_AF'].mean()) / df['length_AF'].std()


,SGDID,Uniprot,SC,length_AF,entangled,asphericity,fraction_idp,Knot,cov_lasso,entangled_residues,Cutsite_residues,clustered_entangled_residues,length_z
0,S000002468,Q12298,0,539,1,0.139968,0.011132,0,0,"[26, 27, 28, 31, 33, 34, 35, 36, 37, 38, 39, 4...",[],"[27, 31, 34, 35, 36, 38, 39, 40, 41, 42, 43, 4...",0.002089
1,S000000094,P35842,0,467,1,0.097823,0.051392,0,0,"[21, 23, 30, 31, 32, 33, 34, 35, 37, 38, 41, 4...",[],"[32, 33, 34, 35, 37, 38, 41, 46, 47, 67, 68, 6...",-0.199317
2,S000002490,P38961,0,392,1,0.108081,0.295918,0,0,"[177, 178, 179, 180, 181, 182, 183, 184, 190, ...",[],"[181, 193, 266, 267, 268, 290, 292, 293, 294, ...",-0.409115
3,S000003653,P46956,0,311,1,0.311604,0.099678,0,0,"[64, 65, 68, 69, 70, 71, 72, 73, 74, 75, 78, 1...",[],"[203, 210, 211, 212, 213, 252, 253, 254, 255, ...",-0.635697
5,S000003242,P53204,0,395,1,0.102152,0.293671,0,0,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 15, 16, 17...",[],"[1, 2, 3, 4, 5, 6, 7, 8, 9, 15, 16, 17, 18, 19...",-0.400723
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2249,S000004850,Q05029,0,724,1,0.111921,0.084254,0,0,"[5, 6, 7, 8, 9, 10, 11, 12, 15, 16, 18, 19, 23...",[],"[6, 7, 8, 9, 10, 11, 12, 15, 35, 37, 38, 39, 4...",0.519590
2250,S000005337,P53743,0,316,1,0.212209,0.506329,0,0,"[103, 104, 105, 106, 107, 108, 109, 110, 111, ...",[],"[103, 104, 105, 106, 107, 108, 109, 110, 111, ...",-0.621710
2251,S000001943,P43619,0,295,1,0.383145,0.071186,0,0,"[1, 2, 3, 4, 5, 8, 10, 12, 13, 14, 15, 16, 17,...",[],"[15, 18, 19, 20, 21, 22, 23, 24, 25, 26, 48, 4...",-0.680454
2254,S000005185,P11412,1,505,1,0.142002,0.017822,0,0,"[6, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, ...","[394, 395, 396, 397, 398, 399, 400, 401, 402]","[10, 11, 12, 13, 14, 15, 16, 17, 18, 23, 24, 2...",-0.093020


In [5]:
records = []
for _, row in df.iterrows():
    uniprot = row['Uniprot']
    length = row['length_AF']
    length_z = row['length_z']
    entangled = set(row['clustered_entangled_residues'])
    cutsites = set(row['Cutsite_residues'])
    # sasa_values = row['sasa_per_residue']

    for i in range(1, length+1):
        res_id = f"{uniprot}_{i}"
        is_entangled = int(i in entangled)
        is_cutsite = int(i in cutsites)
        length_z = length_z
        # sasa = sasa_values[i-1]
        
        records.append({
            'Uniprot_ResID': res_id,
            'is_entangled': is_entangled,
            'is_cutsite': is_cutsite,
            'length_z': length_z
        })

In [6]:
df_residue_level = pd.DataFrame(records)
df_residue_level

,Uniprot_ResID,is_entangled,is_cutsite,length_z
0,Q12298_1,0,0,0.002089
1,Q12298_2,0,0,0.002089
2,Q12298_3,0,0,0.002089
3,Q12298_4,0,0,0.002089
4,Q12298_5,0,0,0.002089
...,...,...,...,...
907490,P41819_314,0,0,-0.616116
907491,P41819_315,0,0,-0.616116
907492,P41819_316,0,0,-0.616116
907493,P41819_317,0,0,-0.616116


# Get OR using contingency table

In [7]:
# contingency table
ent_sc = len(df_residue_level[(df_residue_level['is_entangled']==1) & (df_residue_level['is_cutsite']==1)])
nent_sc = len(df_residue_level[(df_residue_level['is_entangled']==0) & (df_residue_level['is_cutsite']==1)])
ent_nsc = len(df_residue_level[(df_residue_level['is_entangled']==1) & (df_residue_level['is_cutsite']==0)])
nent_nsc = len(df_residue_level[(df_residue_level['is_entangled']==0) & (df_residue_level['is_cutsite']==0)])

print("Contingency table (rows: entanglement, columns: SC)")
print(f"{'':<15} {'SC':>10} {'Non-SC':>10}")
print(f"{'Entangled':<15} {ent_sc:>10} {ent_nsc:>10}")
print(f"{'Non-entangled':<15} {nent_sc:>10} {nent_nsc:>10}")

print('-'*50)
table = [[ent_sc, nent_sc],
        [ent_nsc, nent_nsc]]
print(table)
ct = Table2x2(table)

odds_ratio, pvalue = ss.fisher_exact(table)
ci_low, ci_high = ct.oddsratio_confint(alpha=0.05, method="exact")

print(f"Odds Ratio: {odds_ratio:.3f}")
print(f"95% CI: [{ci_low:.3f}, {ci_high:.3f}]")
print(f"P-value: {pvalue}")

Contingency table (rows: entanglement, columns: SC)
                        SC     Non-SC
Entangled             2027     203837
Non-entangled         4277     697354
--------------------------------------------------
[[2027, 4277], [203837, 697354]]
Odds Ratio: 1.621
95% CI: [1.538, 1.710]
P-value: 5.62984220126421e-67


In [8]:
print("The prevalence of age-associated structural changes across the set of entangled protein residues:")
print(f"structural change: {ent_sc + nent_sc}")
print(f"Total residues: {ent_sc+ent_nsc+nent_sc+nent_nsc}")
print(f"Percentage: {100*(ent_sc + nent_sc)/(ent_sc+ent_nsc+nent_sc+nent_nsc):.3f} %")

The prevalence of age-associated structural changes across the set of entangled protein residues:
structural change: 6304
Total residues: 907495
Percentage: 0.695 %


In [9]:
print("Residues composing entangled regions exhibit a structural change:")
print(f"Entangled residues exhibit structural change: {ent_sc}")
print(f"Total Entangled residues: {(ent_sc+ent_nsc)}")
print(f"Percentage: {100*(ent_sc)/(ent_sc+ent_nsc):.3f} %")

Residues composing entangled regions exhibit a structural change:
Entangled residues exhibit structural change: 2027
Total Entangled residues: 205864
Percentage: 0.985 %


In [10]:
print("Residues NOT composing entangled regions exhibit a structural change:")
print(f"Non-entangled residues exhibit Structural change: {nent_sc}")
print(f"Total Non-entangled residues: {(nent_sc+nent_nsc)}")
print(f"Percentage: {100*(nent_sc)/(nent_sc+nent_nsc):.3f} %")

Residues NOT composing entangled regions exhibit a structural change:
Non-entangled residues exhibit Structural change: 4277
Total Non-entangled residues: 701631
Percentage: 0.610 %


# Logistic Regression

In [11]:
model =smf.logit('is_cutsite ~ is_entangled + length_z', data=df_residue_level)
result = model.fit()
print(result.summary())

Optimization terminated successfully.
         Current function value: 0.040367
         Iterations 10
                           Logit Regression Results                           
Dep. Variable:             is_cutsite   No. Observations:               907495
Model:                          Logit   Df Residuals:                   907492
Method:                           MLE   Df Model:                            2
Date:                Mon, 02 Mar 2026   Pseudo R-squ.:                 0.02598
Time:                        15:21:28   Log-Likelihood:                -36633.
converged:                       True   LL-Null:                       -37610.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       -4.9335      0.015   -321.039      0.000      -4.964      -4.903
is_entangled     0.461

In [12]:
summary_df = summarize_logit(result)
print(summary_df.to_string(float_format=lambda x: f"{x:.3f}"))

               coef    OR  OR_ci_lower  OR_ci_upper  pvalue       p_fmt
Intercept    -4.933 0.007        0.007        0.007   0.000   0.000e+00
is_entangled  0.462 1.587        1.505        1.673   0.000   4.812e-65
length_z     -0.503 0.605        0.588        0.622   0.000  8.459e-267


# calculate the odds

In [13]:
# Odd of residues in entangled region to exhibit alter proteolytic assessibility
# Coefficients and covariance matrix
params = result.params
cov = result.cov_params()

# Example: calculate odds and CI for entangled = 1, length_z = 0
a = np.array([1, 1, 0])  # intercept, entangled, length_z

# Compute logit
logit = np.dot(a, params)

# Variance and standard error
var_logit = np.dot(a, np.dot(cov, a))
se_logit = np.sqrt(var_logit)

# CI in logit scale
logit_lower = logit - 1.96 * se_logit
logit_upper = logit + 1.96 * se_logit

# Convert to odds
odds = np.exp(logit)
ci_lower = np.exp(logit_lower)
ci_upper = np.exp(logit_upper)

print(f"Odds: {odds:.5f} (95% CI: [{ci_lower:.5f}, {ci_upper:.5f}]")

Odds: 0.01143 (95% CI: [0.01094, 0.01194]


In [14]:
# Odd of residues in non-entangled region to exhibit alter proteolytic assessibility
# Coefficients and covariance matrix
params = result.params
cov = result.cov_params()

# Example: calculate odds and CI for entangled = 0, length_z = 0
a = np.array([1, 0, 0])  # intercept, entangled, length_z

# Compute logit
logit = np.dot(a, params)

# Variance and standard error
var_logit = np.dot(a, np.dot(cov, a))
se_logit = np.sqrt(var_logit)

# CI in logit scale
logit_lower = logit - 1.96 * se_logit
logit_upper = logit + 1.96 * se_logit

# Convert to odds
odds = np.exp(logit)
ci_lower = np.exp(logit_lower)
ci_upper = np.exp(logit_upper)

print(f"Odds: {odds:.5f} (95% CI: [{ci_lower:.5f}, {ci_upper:.5f}]")

Odds: 0.00720 (95% CI: [0.00699, 0.00742]
